# Detector v2 (CenterNet r34, thạch bản + STT) — huấn luyện trên Kaggle · Run All · reset-safe

**Trước khi chạy:** Settings → Accelerator **GPU T4** (x2 cũng được; P100 có thể lỗi với torch mới) · Persistence **Files only**
(giữ `/kaggle/working/last.pt` khi restart) · Add Input → dataset dựng từ `i5v2_bundle.zip` (`make_bundle.py`).
Internet chỉ cần nếu đẩy lên HF (`HF_REPO` + Secret `HF_TOKEN`).

Mỗi epoch: train cân bằng 50/50 thạch bản/STT (aug nền xám · otsu/stretch · kéo dọc ±10 % · nét · blur · crop cột) →
đánh giá trên **val page-disjoint**: ok50 / miss / extra / |dy| / **% tầng n==N** (hộp thô @0,15 & 0,2) / cắt thân chữ,
**STT F1** (guard ≥ v1 − 0,01), Chrestomathie (sách không train). Best theo ok50_litho qua guard; dừng sớm 4 epoch.
Kết quả: `/kaggle/working/{best.pt, last.pt, metrics.csv, report.md}`. **Reset?** Run All lại → tự resume từ `last.pt`.

In [3]:
# ---- 1) Cấu hình — chỉ sửa ở đây -------------------------------------------------
EPOCHS   = 20        # ≈ 2–3 phút/epoch trên T4 ở img 1024 (20 epoch ≈ 1 giờ + eval)
BATCH    = 4         # hết RAM GPU -> 2
IMG      = 1024      # hoặc 1280 (chậm ~1,6×, batch 2)
LR       = 2e-4      # cosine + warmup 1 epoch
SEED     = 0
EVAL_CHRESTO = 20    # số trang Chrestomathie (held-out, không train) đo mỗi epoch; 0 = bỏ
HF_REPO  = "mdnt571/nom-char-det-v2"        # vd "mdnt571/nom-char-det-v2": đẩy last/best mỗi epoch + resume từ hub (cần Secret HF_TOKEN). "" = local
EXTRA    = ""        # tham số thêm cho train_kaggle.py, vd "--patience 6 --stt-tol 0.01 --p-gray 0.5"
SMOKE    = False     # True: chạy thử 4 trang/miền, 1 epoch (kiểm tra đường ống ~2 phút)

In [4]:
# ---- 2) Tìm bundle, chép mã ra /kaggle/working, kiểm GPU ---------------------------
import os, sys, glob, shutil, subprocess
hits = sorted(glob.glob("/kaggle/input/**/train_kaggle.py", recursive=True)) or sorted(glob.glob("./**/train_kaggle.py", recursive=True))
assert hits, "Không thấy train_kaggle.py — Add Input: gắn dataset dựng từ i5v2_bundle.zip"
DATA = os.path.dirname(hits[0])
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
CODE = os.path.join(WORK, "i5v2_code"); os.makedirs(CODE, exist_ok=True)
shutil.copytree(os.path.join(DATA, "i5v2"), os.path.join(CODE, "i5v2"), dirs_exist_ok=True)
shutil.copy2(os.path.join(DATA, "train_kaggle.py"), os.path.join(CODE, "train_kaggle.py"))
for f in ("manifest_train.json", "manifest_val.json", "v1/detector_r34.best.pt", "bundle_stats.json"):
    assert os.path.exists(os.path.join(DATA, f)), f"bundle thiếu {f}"
tok = ""
if HF_REPO:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "huggingface_hub"], check=False)
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = os.environ.get("HF_TOKEN", "")
    os.environ["HF_TOKEN"] = tok or ""
import torch, json
print("data :", DATA); print("code :", CODE); print("out  :", WORK)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (bật GPU T4!)", "| torch", torch.__version__)
print("bundle:", json.load(open(os.path.join(DATA, "bundle_stats.json")))["by_domain_split"])
print("HF   :", HF_REPO or "(local-only)", "| token", "có" if tok else "không")
print("resume:", "có last.pt → tiếp tục" if os.path.exists(os.path.join(WORK, "last.pt")) else "chưa có last.pt → từ đầu")

data : /kaggle/input/datasets/truongmdn/i5-detector-v2
code : /kaggle/working/i5v2_code
out  : /kaggle/working
GPU  : Tesla T4 | torch 2.10.0+cu128
bundle: {'litho/train': 215, 'stt/train': 356, 'litho/val': 27, 'stt/val': 45, 'litho/test': 26, 'stt/test': 44, 'chresto/chresto': 64}
HF   : mdnt571/nom-char-det-v2 | token có
resume: chưa có last.pt → từ đầu


In [5]:
# ---- 3) Train — reset-safe (chạy lại cell này/Run All sau reset để tiếp tục) --------
cmd = [sys.executable, os.path.join(CODE, "train_kaggle.py"), "--data", DATA, "--out", WORK,
       "--epochs", str(EPOCHS), "--batch", str(BATCH), "--img", str(IMG), "--lr", str(LR), "--seed", str(SEED),
       "--eval-chresto", str(EVAL_CHRESTO), "--workers", "2"]
if HF_REPO: cmd += ["--hf-repo", HF_REPO]
if SMOKE:   cmd += ["--smoke"]
if EXTRA:   cmd += EXTRA.split()
print("$", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

$ /usr/bin/python3 /kaggle/working/i5v2_code/train_kaggle.py --data /kaggle/input/datasets/truongmdn/i5-detector-v2 --out /kaggle/working --epochs 20 --batch 4 --img 1024 --lr 0.0002 --seed 0 --eval-chresto 20 --workers 2 --hf-repo mdnt571/nom-char-det-v2
[v2] device=cuda amp=True | train {'litho': 215, 'stt': 356} → 430 mẫu/epoch (107 bước × batch 4) | val litho 27 stt 45 chresto 20 | img 1024 | init /kaggle/input/datasets/truongmdn/i5-detector-v2/v1/detector_r34.best.pt
  mốc v1 (11s): ok50 98.71 | n==N@0,15 94.7 | cut 11.32 | STT F1@0,2 0.8772 (@0,15 0.876) | chresto n==N 83.8
[hf] không có last.pt trên hub (RemoteEntryNotFoundError)
[hf] không có best.pt trên hub (RemoteEntryNotFoundError)
    ep 1 bước    0/107 lr 1.87e-06 loss 2.9817 (hm 2.0414 wh 4.4145 off 0.4989)
    ep 1 bước   20/107 lr 3.93e-05 loss 1.9396 (hm 1.2988 wh 1.4850 off 0.4924)
    ep 1 bước   40/107 lr 7.66e-05 loss 1.7929 (hm 1.0107 wh 3.1532 off 0.4669)
    ep 1 bước   60/107 lr 1.14e-04 loss 1.7971 (hm 1.1436

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  /kaggle/working/last.pt     :   1%|          | 2.26MB /  346MB            

Processing Files (0 / 1)      :   1%|          | 2.26MB /  346MB, 3.76MB/s  
New Data Upload               :   2%|▏         | 2.26MB /  134MB, 3.76MB/s  

Processing Files (0 / 1)      :   6%|▌         | 19.2MB /  346MB, 24.0MB/s  
New Data Upload               :  14%|█▍        | 19.2MB /  134MB, 24.0MB/s  

Processing Files (0 / 1)      :  13%|█▎        | 44.6MB /  346MB, 44.6MB/s  
New Data Upload               :  33%|███▎      | 44.6MB /  134MB, 44.6MB/s  

Processing Files (0 / 1)      :  19%|█▉        | 66.6MB /  346MB, 55.5MB/s  
New Data Upload               :  50%|████▉     | 66.6MB /  134MB, 55.5MB/s  

  /kaggle/working/last.pt     :  19%|█▉        | 66.6MB /  346MB            

  /kaggle/working/last.pt     :  19%|█▉        | 66.6MB /  346MB            


[hf] đẩy last.pt -> mdnt571/nom-char-det-v2
[hf] đẩy metrics.csv -> mdnt571/nom-char-det-v2
[hf] đẩy report.md -> mdnt571/nom-char-det-v2
    ep 2 bước    0/107 lr 2.00e-04 loss 1.5330 (hm 0.9301 wh 1.6100 off 0.4419)
    ep 2 bước   20/107 lr 2.00e-04 loss 1.3516 (hm 0.7228 wh 1.8337 off 0.4454)
    ep 2 bước   40/107 lr 2.00e-04 loss 1.6306 (hm 1.0101 wh 1.5395 off 0.4665)
    ep 2 bước   60/107 lr 2.00e-04 loss 1.6552 (hm 1.0752 wh 1.4649 off 0.4335)
    ep 2 bước   80/107 lr 1.99e-04 loss 0.9075 (hm 0.3640 wh 1.5296 off 0.3905)
    ep 2 bước  100/107 lr 1.99e-04 loss 1.6332 (hm 1.0185 wh 1.6986 off 0.4448)
epoch  2/20   44.3s loss 1.5615 | litho ok50 100.0 miss 0.0 extra 0.72 n==N@0,15 97.8 (@0,2 98.5) cut 0.37 | STT F1@0,2 0.8639 (TỤT) | chresto n==N 73.7


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  /kaggle/working/last.pt     :   0%|          | 70.8kB /  346MB            

Processing Files (0 / 1)      :   0%|          | 70.8kB /  346MB,   ???B/s  

  /kaggle/working/last.pt     :   0%|          | 70.8kB /  346MB            

Processing Files (0 / 1)      :   1%|          | 4.02MB /  346MB, 9.86MB/s  
New Data Upload               :   3%|▎         | 3.95MB /  134MB, 9.86MB/s  

Processing Files (0 / 1)      :   4%|▍         | 14.2MB /  346MB, 23.5MB/s  
New Data Upload               :  11%|█         | 14.1MB /  134MB, 23.5MB/s  

Processing Files (0 / 1)      :   8%|▊         | 26.6MB /  346MB, 33.1MB/s  
New Data Upload               :  20%|█▉        | 26.5MB /  134MB, 33.1MB/s  

Processing Files (0 / 1)      :  15%|█▌        | 52.0MB /  346MB, 51.9MB/s  
New Data Upload               :  39%|███▊      | 51.9MB /  134MB, 51.9MB/s  


[hf] đẩy last.pt -> mdnt571/nom-char-det-v2
[hf] đẩy metrics.csv -> mdnt571/nom-char-det-v2
[hf] đẩy report.md -> mdnt571/nom-char-det-v2
    ep 3 bước    0/107 lr 1.99e-04 loss 1.4237 (hm 0.8481 wh 1.8115 off 0.3945)
    ep 3 bước   20/107 lr 1.98e-04 loss 1.3360 (hm 0.7429 wh 1.7287 off 0.4202)
    ep 3 bước   40/107 lr 1.97e-04 loss 1.5724 (hm 0.9239 wh 2.3376 off 0.4148)
    ep 3 bước   60/107 lr 1.97e-04 loss 1.5991 (hm 0.9926 wh 1.7424 off 0.4322)
    ep 3 bước   80/107 lr 1.96e-04 loss 1.0157 (hm 0.4812 wh 1.3665 off 0.3979)
    ep 3 bước  100/107 lr 1.95e-04 loss 1.3163 (hm 0.7092 wh 1.8470 off 0.4224)
epoch  3/20   49.0s loss 1.4831 | litho ok50 100.0 miss 0.0 extra 0.85 n==N@0,15 98.7 (@0,2 98.9) cut 0.22 | STT F1@0,2 0.8584 (TỤT) | chresto n==N 74.7


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  /kaggle/working/last.pt     :   0%|          | 70.8kB /  346MB            

Processing Files (0 / 1)      :   0%|          | 70.8kB /  346MB,   ???B/s  

Processing Files (0 / 1)      :   1%|          | 4.02MB /  346MB, 19.8MB/s  
New Data Upload               :   6%|▌         | 3.95MB / 67.0MB, 19.8MB/s  

  /kaggle/working/last.pt     :   1%|          | 4.02MB /  346MB            

Processing Files (0 / 1)      :   6%|▌         | 19.2MB /  346MB, 32.0MB/s  
New Data Upload               :  14%|█▍        | 19.2MB /  134MB, 32.0MB/s  

Processing Files (0 / 1)      :  13%|█▎        | 44.6MB /  346MB, 55.7MB/s  
New Data Upload               :  33%|███▎      | 44.5MB /  134MB, 55.7MB/s  

Processing Files (0 / 1)      :  18%|█▊        | 63.2MB /  346MB, 63.1MB/s  
New Data Upload               :  47%|████▋     | 63.1MB /  134MB, 63.1MB/s  


[hf] đẩy last.pt -> mdnt571/nom-char-det-v2
[hf] đẩy metrics.csv -> mdnt571/nom-char-det-v2
[hf] đẩy report.md -> mdnt571/nom-char-det-v2
    ep 6 bước    0/107 lr 1.79e-04 loss 0.9271 (hm 0.4091 wh 1.2703 off 0.3909)
    ep 6 bước   20/107 lr 1.77e-04 loss 1.4280 (hm 0.8855 wh 1.1834 off 0.4242)
    ep 6 bước   40/107 lr 1.75e-04 loss 1.4408 (hm 0.8918 wh 1.2158 off 0.4274)
    ep 6 bước   60/107 lr 1.73e-04 loss 1.6692 (hm 1.1089 wh 1.0211 off 0.4582)
    ep 6 bước   80/107 lr 1.71e-04 loss 1.2916 (hm 0.7454 wh 1.2889 off 0.4174)
    ep 6 bước  100/107 lr 1.69e-04 loss 2.2101 (hm 1.5014 wh 2.2798 off 0.4807)
epoch  6/20   48.9s loss 1.3857 | litho ok50 100.0 miss 0.0 extra 0.53 n==N@0,15 98.0 (@0,2 98.2) cut 0.22 | STT F1@0,2 0.8625 (TỤT) | chresto n==N 73.7


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  /kaggle/working/last.pt     :   0%|          | 70.8kB /  346MB            

Processing Files (0 / 1)      :   0%|          | 70.8kB /  346MB,   ???B/s  

  /kaggle/working/last.pt     :   0%|          | 70.8kB /  346MB            

Processing Files (0 / 1)      :   1%|          | 4.02MB /  346MB, 9.85MB/s  
New Data Upload               :   3%|▎         | 3.95MB /  134MB, 9.85MB/s  

Processing Files (0 / 1)      :   2%|▏         | 7.40MB /  346MB, 12.2MB/s  
New Data Upload               :   5%|▌         | 7.33MB /  134MB, 12.2MB/s  

Processing Files (0 / 1)      :   8%|▊         | 28.8MB /  346MB, 35.9MB/s  
New Data Upload               :  21%|██▏       | 28.8MB /  134MB, 35.9MB/s  

Processing Files (0 / 1)      :  14%|█▍        | 49.7MB /  346MB, 49.6MB/s  
New Data Upload               :  37%|███▋      | 49.6MB /  134MB, 49.6MB/s  


[hf] đẩy last.pt -> mdnt571/nom-char-det-v2
[hf] đẩy metrics.csv -> mdnt571/nom-char-det-v2
[hf] đẩy report.md -> mdnt571/nom-char-det-v2
  dừng sớm: 4 epoch không cải thiện ≥ 0.2 điểm ok50 (hoặc STT tụt)
[done] 349s | best epoch -1 score(ok50,n==N,-cut) (98.7, 94.7, -11.32) (v1 (98.7, 94.7, -11.32)) | KHÔNG có best.pt: không epoch nào vượt v1 qua guard STT -> giữ v1 (+ detector_resize area) | /kaggle/working/report.md


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/i5v2_code/train_kaggle.py', '--data', '/kaggle/input/datasets/truongmdn/i5-detector-v2', '--out', '/kaggle/working', '--epochs', '20', '--batch', '4', '--img', '1024', '--lr', '0.0002', '--seed', '0', '--eval-chresto', '20', '--workers', '2', '--hf-repo', 'mdnt571/nom-char-det-v2'], returncode=0)

In [6]:
# ---- 4) Kết quả: bảng v1 vs v2 + theo epoch ------------------------------------------
import csv
print(open(os.path.join(WORK, "report.md"), encoding="utf-8").read())
rows = list(csv.DictReader(open(os.path.join(WORK, "metrics.csv"), encoding="utf-8")))
keys = ["epoch", "loss", "litho_ok50", "litho_tiers_eq_015", "litho_tiers_eq_020", "litho_cut", "stt_F1_020", "chresto_tiers_eq_015", "is_best"]
print(" | ".join(keys))
for r in rows:
    print(" | ".join(str(r.get(k, "")) for k in keys))
bp = os.path.join(WORK, "best.pt")
if os.path.exists(bp):
    d = torch.load(bp, map_location="cpu", weights_only=False)
    print("\nbest.pt: epoch", d["epoch"], "| img", d["img"], "| use_dcn", d["use_dcn"], "|", os.path.getsize(bp) // 2**20, "MB")
    print("val:", {k: d["val"].get(k) for k in ("litho_ok50", "litho_tiers_eq_015", "litho_cut", "stt_F1_020")})
    print("v1 :", {k: d["base_v1"].get(k) for k in ("litho_ok50", "litho_tiers_eq_015", "litho_cut", "stt_F1_020")})
else:
    print("\nKHÔNG có best.pt: không epoch nào qua guard STT F1 ≥ v1 − 0,01 → giữ v1 (xem report.md)")

# Detector v2 — báo cáo huấn luyện (2026-09-22 14:30)

img 1024 · batch 4 · lr 0.0002 · epochs chạy 6/20 · best epoch **-1** · init `detector_r34.best.pt` · seed 0 · smoke False

Val page-disjoint (trang 10k+3 mỗi sách). Mọi số là proxy (ô tham chiếu = kim + chiếu mực, không GT người); % tầng n==N đếm hộp THÔ (không ép N). Chrestomathie không có trang nào trong train.

| chỉ số | v1 | v2 (best) | Δ | mục tiêu |
|---|---|---|---|---|
| thạch bản val: ô tham chiếu IoU ≥ 0,5 (%) | 98.71 | None |  ↑ | ≥ 98 |
| thạch bản: miss (%) | 0.6 | None |  ↓ |  |
| thạch bản: extra / 100 ô | 0.66 | None |  ↓ |  |
| thạch bản: |dy| tâm trung vị (% bước) | 5.6 | None |  ↓ |  |
| thạch bản: |dy| p90 (% bước) | 14.0 | None |  ↓ |  |
| thạch bản: % tầng n == N @0,15 (hộp thô) | 94.7 | None |  ↑ | ≥ 90 |
| thạch bản: % tầng n == N @0,2 | 88.6 | None |  ↑ |  |
| thạch bản: % tầng n < N @0,15 | 1.8 | None |  ↓ |  |
| thạch bản: % tầng n > N @0,15 | 3.5 | None |  ↓ |  |
| thạch bản: cắt vào thân chữ (%; không

### Xong
- Tải **`best.pt`** (tab Output / `/kaggle/working/best.pt`) + `metrics.csv` + `report.md` về máy.
- Ở repo: `lab/i5_detector_v2/apply_v2.sh <đường dẫn best.pt>` → đo `box_ref_eval` v1↔v2 trên ảnh gốc, build `--book all-new --suffix _v2`.
- Reset / hết giờ: **Run All** lại — cell (3) tự resume từ `last.pt` (Persistence Files only) hoặc từ HF nếu khai `HF_REPO`.